# Task 5: Exploratory Data Analysis (EDA) — Titanic Dataset

**Internship task:** Data Analyst Internship  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Goal:** Explore the dataset, identify patterns and anomalies, and communicate findings.

> Run the notebook from top to bottom. The first code cell downloads the public Titanic dataset and saves `titanic.csv` in this project folder. An internet connection is needed for the first run.

## 1. Business questions
- What proportion of passengers survived?
- How did survival differ by sex and passenger class?
- What do age and fare distributions look like?
- Which numeric variables are correlated?
- What data-quality issues should be considered before modeling?


In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / 'titanic.csv'
DATA_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

if not DATA_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
sns.set_theme(style='whitegrid', palette='deep')
print('Dataset loaded successfully:', df.shape)
display(df.head())


## 2. Understand the data
Use `.shape`, `.info()`, `.describe()`, and `.value_counts()` to inspect the dataset and its distributions.


In [ ]:
print('Rows, columns:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types and non-null counts:')
df.info()
print('\nDescriptive statistics:')
display(df.describe(include='all').T)
print('\nSurvival counts:')
display(df['survived'].value_counts(dropna=False).rename(index={0:'Did not survive', 1:'Survived'}))
print('\nSex counts:')
display(df['sex'].value_counts(dropna=False))
print('\nPassenger class counts:')
display(df['class'].value_counts(dropna=False))


## 3. Data quality and missing values
Missing data is not automatically an error. Quantify it before deciding how to handle it. This notebook focuses on exploration, so it does not silently impute missing values.


In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
}).sort_values('missing_percent', ascending=False)
display(missing[missing['missing_count'] > 0])

print('Duplicate rows:', df.duplicated().sum())
print('Unique values per column:')
display(df.nunique(dropna=False).sort_values(ascending=False).to_frame('unique_values'))


## 4. Univariate analysis
Univariate analysis studies one variable at a time. Histograms show distribution shape; count plots show category frequencies; boxplots help flag potential outliers.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(data=df, x='age', bins=25, kde=True, ax=axes[0,0])
axes[0,0].set_title('Passenger Age Distribution')
sns.histplot(data=df, x='fare', bins=30, kde=True, ax=axes[0,1])
axes[0,1].set_title('Ticket Fare Distribution')
sns.countplot(data=df, x='sex', ax=axes[1,0])
axes[1,0].set_title('Passenger Count by Sex')
sns.countplot(data=df, x='class', order=['First','Second','Third'], ax=axes[1,1])
axes[1,1].set_title('Passenger Count by Class')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 2.8))
sns.boxplot(data=df, x='fare')
plt.title('Fare Boxplot — Potential Outliers')
plt.show()


### Observations to write after viewing the charts
- Age has missing observations; interpret its distribution with that limitation in mind.
- Fare is typically right-skewed: most passengers paid lower fares, while a smaller number paid much higher fares.
- Compare class counts rather than assuming the three classes have equal representation.
- Points beyond a boxplot whisker are potential outliers, not automatic errors. High fares may reflect premium tickets or group bookings.


## 5. Bivariate analysis
Bivariate analysis compares two variables. Here we compare survival with sex, passenger class, and age.


In [ ]:
survival_rate = df.groupby('sex', observed=False)['survived'].mean().mul(100).round(1)
display(survival_rate.rename('survival_rate_percent').to_frame())

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='sex', y='survived', errorbar=None)
plt.ylabel('Survival rate')
plt.title('Survival Rate by Sex')
plt.ylim(0, 1)
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='class', y='survived', order=['First','Second','Third'], errorbar=None)
plt.ylabel('Survival rate')
plt.title('Survival Rate by Passenger Class')
plt.ylim(0, 1)
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='survived', y='age')
plt.xticks([0, 1], ['Did not survive', 'Survived'])
plt.title('Age Distribution by Survival Outcome')
plt.show()

print('Survival rate by class (%):')
display((df.groupby('class', observed=False)['survived'].mean()*100).round(1).rename('survival_rate_percent').to_frame())


### Interpretation
- Compare the survival-rate bars rather than only comparing raw survivor counts; groups have different sizes.
- In the commonly used Titanic dataset, survival rates are substantially higher among female passengers and first-class passengers. Treat these as associations, not proof that one variable alone caused survival.
- Age distributions overlap, so age alone does not cleanly separate survivors from non-survivors.


## 6. Multivariate analysis and correlation
A correlation heatmap summarizes linear relationships among numeric variables. A pairplot shows pairwise distributions and relationships for selected variables. Correlation does not establish causation.


In [ ]:
numeric = df.select_dtypes(include=np.number)
corr = numeric.corr(numeric_only=True)
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap — Numeric Variables')
plt.tight_layout()
plt.show()

pair_cols = [c for c in ['survived', 'age', 'fare', 'pclass', 'sibsp', 'parch'] if c in df.columns]
sns.pairplot(df[pair_cols].dropna().sample(min(350, len(df.dropna(subset=pair_cols))), random_state=42), diag_kind='hist')
plt.show()


## 7. Grouped analysis and concise findings
The following table gives a compact view of survival rates by sex and class. Values are calculated from the dataset when the notebook is run.


In [ ]:
grouped = (df.groupby(['sex', 'class'], observed=False)['survived']
             .agg(passengers='size', survival_rate='mean')
             .assign(survival_rate_percent=lambda x: (x['survival_rate']*100).round(1))
             .drop(columns='survival_rate')
             .sort_values('survival_rate_percent', ascending=False))
display(grouped)

overall_survival = (df['survived'].mean() * 100).round(1)
print(f'Overall survival rate: {overall_survival}%')
print(f'Rows: {len(df):,}; columns: {df.shape[1]}')
print('Highest missingness columns:')
display(missing.head(5))


## 8. Conclusion
The EDA suggests that passenger survival was associated with sex and passenger class. Age and fare have uneven distributions, and missing values—especially in some columns—must be addressed before modeling. The next step for a predictive project would be to define a leakage-safe train/test workflow, decide how to handle missing values, encode categorical features, and evaluate a baseline model. These exploratory patterns should not be interpreted as causal claims.

## 9. Interview questions — short answers
1. **What is EDA and why is it important?** Exploratory Data Analysis uses summary statistics and visualizations to understand data quality, distributions, relationships, and anomalies before modeling or decision-making.
2. **Which plots check correlation?** A correlation heatmap is useful for many numeric variables; scatterplots show relationships between two numeric variables; pairplots show multiple pairwise relationships.
3. **How do you handle skewed data?** First inspect the cause and business meaning. Depending on the task, use a log or other transformation, robust statistics, suitable models, or report median/IQR alongside mean/standard deviation. Do not transform automatically.
4. **How do you detect multicollinearity?** Inspect a correlation matrix and calculate Variance Inflation Factor (VIF) for model predictors. High pairwise correlation is a warning, but multicollinearity can involve multiple variables.
5. **Univariate, bivariate, multivariate analysis?** One variable; two variables; and three or more variables, respectively.
6. **Heatmap vs pairplot?** A heatmap encodes a matrix of values (such as correlations) with color; a pairplot displays pairwise scatterplots and univariate distributions.
7. **How do you summarize insights?** State the finding, quantify it, mention the relevant segment or comparison, explain limitations, and give a practical implication.

---
**Reproducibility note:** This notebook calculates its numerical outputs directly from the downloaded CSV. Use the outputs from your own run when writing final figures in the report.
